In [ ]:
import pyreadr
import torch

from src.gaussian_mixture import GaussianMixtureMAR
from src.metrics import energy_distance_faster

import numpy as np
import matplotlib.pyplot as plt
import math

Load dataset.

In [ ]:

k_range = range(1,200,4)

datasets = ["parkinsons", "allergens", "concrete", "windspeed", "forest", "housing", "stock", "pumadyn32nm", "scm20d", "scm1d"]
i = 1 # index of the dataset to use

Xstar_df = pyreadr.read_r(f"datasets/split_test/test.{datasets[i]}.RDS")[None]
X_miss_df = pyreadr.read_r(f"datasets/split_amputed/mar.{datasets[i]}.RDS")[None]

Prepare the data.

In [ ]:
Xm = torch.tensor(X_miss_df.values, dtype=torch.float64)
X = torch.nan_to_num(Xm, nan=0.0) # input to GMM

Xstar = torch.tensor(Xstar_df.values)

M_np = (X_miss_df.notna()).astype(int).values
M = torch.tensor(M_np, dtype=torch.float64)

In [ ]:
n_samples = X.shape[0]
crit = 'bic'
ct = 'diag'
n_inits = 20

gmm = GaussianMixtureMAR(
            k_range=k_range, 
            criterion=crit,
            device='cpu', 
            cov_type=ct, 
            n_init=n_inits
        )
gmm.fit(X, M)
print('best k = ', gmm.best_k)

In [ ]:
energy_distance_gmm_scaled = [energy_distance_faster(Xstar, gmm.sample(n_samples)[0], scale=True) for _ in range(50)]

In [ ]:
plt.boxplot(energy_distance_gmm_scaled)
plt.title(f'{datasets[i]}: scaled energy distance')
plt.show()